## Useremo lo SQAD dataset (un dataset di Question Answering) per fine tunare T5 Small sul task specifico (transfer learning)

In [29]:
from datasets import load_dataset

# Carichiamo il dataset SQuAD-it
dataset = load_dataset("squad_it")

# Esempio di accesso:
print(dataset["train"][0]["question"])
print(dataset["train"][0]["context"])
print(dataset["train"][0]["answers"]["text"][0])

In quale anno si è verificato il terremoto nel Sichuan?
Il terremoto del Sichuan del 2008 o il terremoto del Gran Sichuan, misurato a 8.0 Ms e 7.9 Mw, e si è verificato alle 02:28:01 PM China Standard Time all' epicentro (06:28:01 UTC) il 12 maggio nella provincia del Sichuan, ha ucciso 69.197 persone e lasciato 18.222 dispersi.
2008


In [30]:
print(dataset["train"][0])

{'id': '56cdca7862d2951400fa6826', 'context': "Il terremoto del Sichuan del 2008 o il terremoto del Gran Sichuan, misurato a 8.0 Ms e 7.9 Mw, e si è verificato alle 02:28:01 PM China Standard Time all' epicentro (06:28:01 UTC) il 12 maggio nella provincia del Sichuan, ha ucciso 69.197 persone e lasciato 18.222 dispersi.", 'question': 'In quale anno si è verificato il terremoto nel Sichuan?', 'answers': {'text': ['2008'], 'answer_start': [29]}}


In [31]:
def parse_squad(dataset):
    """Extract all the answers/questions pairs from the SQuAD dataset

    Args:
        dataset (dict): The imported JSON dataset

    Returns:
        inputs, targets: Two lists containing the inputs and the targets for the QA model
    """

    inputs, targets = [], []

    
    # Loop over all the articles
    for article in dataset:
            
        # Extract context from the paragraph
        context = article['context']
        # Create the question/context sequence
        question_context = 'question: ' + article['question'] + ' context: ' + context
        
        # Create the answer sequence. Use the text field of the first answer
        answer = 'answer: ' + article['answers']['text'][0]
        
        # Add the question_context to the inputs list
        inputs.append(question_context)
        
        # Add the answer to the targets list
        targets.append(answer)
    
    
    return inputs, targets

In [32]:
from termcolor import colored

inputs_train, targets_train =  parse_squad(dataset['train'])  
inputs_test, targets_test =  parse_squad(dataset['test'])          

print("Number of question/answer pairs: " + str(len(inputs_train)))

print('\nFirst Q/A pair:\n\ninputs: ' + colored(inputs_train[0], 'blue'))
print('\ntargets: ' + colored(targets_train[0], 'green'))
print('\nLast Q/A pair:\n\ninputs: ' + colored(inputs_train[-1], 'blue'))
print('\ntargets: ' + colored(targets_train[-1], 'green'))

Number of question/answer pairs: 54159

First Q/A pair:

inputs: question: In quale anno si è verificato il terremoto nel Sichuan? context: Il terremoto del Sichuan del 2008 o il terremoto del Gran Sichuan, misurato a 8.0 Ms e 7.9 Mw, e si è verificato alle 02:28:01 PM China Standard Time all' epicentro (06:28:01 UTC) il 12 maggio nella provincia del Sichuan, ha ucciso 69.197 persone e lasciato 18.222 dispersi.

targets: answer: 2008

Last Q/A pair:

inputs: question: In quale animale lo zinco è tossico fino al punto di velenosi? context: Pennies e altre monete piccole a volte sono ingerite dai cani, con conseguente necessità di cure mediche per rimuovere il corpo estraneo. Il contenuto di zinco di alcune monete può causare la tossicità dello zinco, che è comunemente mortale nei cani, dove provoca una grave anemia emolitica, e anche danni al fegato o ai reni; vomito e diarrea sono sintomi possibili. Lo zinco è altamente tossico nei pappagalli e l' avvelenamento può spesso essere mortal

In [36]:
print(inputs_train[200])
print('-'*100)
print(targets_train[200])

question: Qual è il costo per il bilancio della provincia? context: Nel 2008, il Consiglio di Stato ha stabilito un piano di sostegno alla controparte????????? Il piano è quello di organizzare 19 province orientali e centrali e municipalitie per aiutare 18 contee, su "una provincia a una contea interessata" base. Il piano ha una durata di 3 anni, e costa non meno dell' uno per cento del bilancio della provincia o del comune. un piano di sostegno di contropartita.
----------------------------------------------------------------------------------------------------
answer: uno per cento


## Carichiamo il modello preaddestrato

In [37]:
from transformers import T5ForConditionalGeneration, T5TokenizerFast

model_name = "gsarti/it5-small" 
tokenizer = T5TokenizerFast.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

## Definiamo dataset e dataloader

In [38]:
import torch
from torch.utils.data import Dataset, DataLoader

class SquadDataset(Dataset):
    def __init__(self, inputs, answers, tokenizer, max_input_len=512, max_target_len=64):
        self.inputs = inputs
        self.answers = answers
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        source_text = self.inputs[idx]
        target_text = self.answers[idx]

        # Tokenizzazione Input
        source = self.tokenizer(
            source_text.lower(),
            max_length=self.max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Tokenizzazione Target
        target = self.tokenizer(
            target_text.lower(),
            max_length=self.max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Labels: sostituiamo i pad_token_id con -100 per ignorarli nella loss
        labels = target["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": source["input_ids"].squeeze(),
            "attention_mask": source["attention_mask"].squeeze(),
            "labels": labels
        }

# Inizializzazione
train_dataset = SquadDataset(inputs_train, targets_train, tokenizer)
test_dataset = SquadDataset(inputs_test, targets_test, tokenizer)
train_loader = DataLoader(
    train_dataset, 
    batch_size=8, 
    shuffle=True, 
    pin_memory=True # Velocizza il trasferimento dati alla GPU
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=8, 
    shuffle=False, 
    pin_memory=True
)

## Proviamo il modello preaddestrato ma non fine tunato sul nostro task:

In [41]:
# Generazione
index=0
device = 'cuda'#torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    generated_ids = model.generate(
        input_ids=test_dataset[index]['input_ids'].unsqueeze(0).to(device),
        attention_mask=test_dataset[index]['attention_mask'].unsqueeze(0).to(device),
        max_length=128
    )

# Decodifica
question = tokenizer.decode(test_dataset[index]['input_ids'],skip_special_tokens=True)
prediction = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f"Domanda + contesto: --- {question}")
print(f"Risposta del modello ): {prediction}")

Domanda + contesto: --- question: quando è iniziata la crisi petrolifera del 1973? context: la crisi petrolifera del 1973 iniziò nell' ottobre 1973 quando i membri dell' organizzazione dei paesi esportatori di petrolio arabo (oapec, composta dai membri arabi dell' opec più egitto e siria) proclamarono un embargo petrolifero. alla fine dell' embargo, nel marzo 1974, il prezzo del petrolio era salito da 3 dollari al barile a quasi 12 dollari a livello mondiale; i prezzi americani erano notevolmente più elevati. l' embargo ha causato una crisi petrolifera, o "shock", con molti effetti a breve e lungo termine sulla politica globale e sull' economia globale. più tardi fu chiamato il "primo shock petrolifero", seguito dalla crisi petrolifera del 1979, definita il "secondo shock petrolifero".
Risposta del modello ): ".  l' embargo petrolifero è stato interrotto. l' embargo è stato interrotto nel 1973. dell petrolifero l' embargo petrolifero l' opec e l' egitto e siria l' arabia saudita l   


## Definiamo il tipo di evaluation

In [40]:
import numpy as np
import evaluate
squad_metric = evaluate.load("squad")

def evaluate_model(model, tokenizer, loader, device):
    model.eval()
    all_preds = []
    all_targets = []
    
    print("Calcolo metriche (EM/F1) sul set di test...")
    for batch in tqdm(loader, leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        with torch.no_grad():
            # Generazione (usiamo i parametri standard per QA)
            outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=64)
        
        # Decodifica
        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        
        # Pulizia label (rimozione -100)
        labels = batch["labels"].cpu().numpy()
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        targets = tokenizer.batch_decode(labels, skip_special_tokens=True)
        
        all_preds.extend(preds)
        all_targets.extend(targets)
        break

    # Formattazione per la metrica SQuAD
    formatted_preds = [{"prediction_text": p, "id": str(i)} for i, p in enumerate(all_preds)]
    formatted_refs = [{"answers": {"answer_start": [0], "text": [t]}, "id": str(i)} for i, t in enumerate(all_targets)]
    
    return squad_metric.compute(predictions=formatted_preds, references=formatted_refs)

Calcolare l'F1-score nel Question Answering è leggermente diverso dal classico calcolo che si fa nei problemi di classificazione, perché qui non lavoriamo su etichette predefinite, ma su insiemi di parole (token).In questo contesto, l'F1-score misura la sovrapposizione tra la risposta predetta dal modello e quella reale. Ecco i passaggi logici che il sistema compie dietro le quinte:1. Tokenizzazione e PuliziaPer prima cosa, sia la risposta del modello che la risposta corretta vengono "normalizzate":Si trasforma tutto in minuscolo.Si rimuove la punteggiatura e gli articoli (a, an, the, il, lo, la...).Si divide il testo in singole parole (token).Esempio:Target: "nella provincia del Sichuan" → {'provincia', 'sichuan'}Predizione: "provincia di Sichuan" → {'provincia', 'sichuan'}2. Calcolo dei componenti (Precision e Recall)Si confrontano i due insiemi di parole per trovare i True Positives (TP), ovvero le parole comuni a entrambe.$$Precision = \frac{\text{Parole comuni}}{\text{Totale parole nella Predizione}}$$$$Recall = \frac{\text{Parole comuni}}{\text{Totale parole nel Target}}$$3. La formula dell'F1L'F1 è la media armonica di queste due misure:$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$Un esempio pratico:Supponiamo che la domanda sia sulla capitale d'Italia:Risposta Corretta (Target): "La città di Roma" → Token: {'città', 'roma'}Risposta Modello (Pred): "Roma" → Token: {'roma'}Parole comuni: 1 (roma)Precision: $1 / 1 = 1.0$ (Tutto quello che ha detto il modello è giusto)Recall: $1 / 2 = 0.5$ (Il modello ha trovato solo la metà delle parole del target)F1-Score: $2 \times \frac{1.0 \times 0.5}{1.0 + 0.5} = \mathbf{0.67}$

## Definiamo il ciclo di training

In [11]:
import time
from tqdm.auto import tqdm

# Configurazione
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
grad_acc_steps = 4 
epochs = 3

print(f"Inizio Fine-tuning (FP32) su {device}...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    pbar = tqdm(train_loader, desc=f"Epoca {epoch}")
    
    for step, batch in enumerate(pbar):
        # Sposta i dati sulla GPU
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Forward pass standard
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss / grad_acc_steps
        # Dobbiamo dividere per gli step di accumulo perchè In PyTorch, quando chiami loss.backward(), i gradienti non vengono sovrascritti, ma sommati a quelli esistenti (grad += new_grad).
        #Se non dividessi per 4 (grad_acc_steps), dopo 4 step i gradienti sarebbero 4 volte più grandi del normale. Dividendo la loss, riportiamo la media matematica al valore corretto, come se avessimo processato tutti i 32 esempi in un colpo solo.
        # Backward pass standard
        loss.backward()
        
        total_loss += loss.item() * grad_acc_steps
        
        # Aggiornamento pesi
        if (step + 1) % grad_acc_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            
        # Aggiorna la barra di progresso con la loss corrente
        if step % 10 == 0:
            pbar.set_postfix({"loss": f"{(total_loss / (step + 1)):.4f}"})

    # --- VALUTAZIONE A FINE EPOCA ---
    # Usiamo la funzione evaluate_model definita in precedenza
    metrics = evaluate_model(model, tokenizer, test_loader, device)
    
    print(f"\n--- RISULTATI EPOCA {epoch} ---")
    print(f"Loss Media: {total_loss / len(train_loader):.4f}")
    print(f"Exact Match (EM): {metrics['exact_match']:.2f}")
    print(f"F1-Score: {metrics['f1']:.2f}")
    print(f"Tempo: {time.time() - start_time:.2f}s")
    print("----------------------------\n")
    
    # Salvataggio checkpoint
    model.save_pretrained(f"./model/it5-squad-standard-ep{epoch}")

Inizio Fine-tuning (FP32) su cpu...


Epoca 0:   0%|          | 0/6770 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [25]:
metrics = evaluate_model(model, tokenizer, test_loader, device)

Calcolo metriche (EM/F1) sul set di test...


  0%|          | 0/952 [00:00<?, ?it/s]

## Proviamo il modello addestrato

In [26]:
# Generazione
index=5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    generated_ids = model.generate(
        input_ids=test_dataset[index]['input_ids'].unsqueeze(0).to(device),
        attention_mask=test_dataset[index]['attention_mask'].unsqueeze(0).to(device),
        max_length=128
    )

# Decodifica
question = tokenizer.decode(test_dataset[index]['input_ids'],skip_special_tokens=True)
prediction = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f"Domanda + contesto: --- {question}")
print(f"Risposta del modello ): {prediction}")

Domanda + contesto: --- question: in quale data henry kissinger ha negoziato un ritiro di truppe israeliane dalla penisola del sinai? context: la crisi ha avuto un forte impatto sulle relazioni internazionali e ha creato una frattura all' interno della nato. alcune nazioni europee e il giappone hanno cercato di dissociarsi dalla politica estera degli stati uniti in medio oriente per evitare di essere presi di mira dal boicottaggio. i produttori arabi di petrolio hanno collegato eventuali cambiamenti politici futuri alla pace tra i belligeranti. per affrontare questo problema, l' amministrazione nixon ha avviato negoziati multilaterali con i combattenti. hanno fatto in modo che israele tornasse dalla penisola del sinai e dalle alture del golan. a partire dal 18 gennaio 1974, il segretario di stato statunitense henry kissinger aveva negoziato il ritiro di una truppa israeliana da alcune parti della penisola del sinai. la promessa di un accordo negoziato tra israele e siria è stata suffic